# 6.1 感知机

机器学习的最终目的是找到一组良好的参数𝜃，使得𝜃表示的数学模型能够很好地从训
练集中学到映射关系𝑓𝜃: 𝒙 → 𝒚,   𝒙,𝒚 ∈ 𝔻train，从而利用训练好的𝑓𝜃
(𝒙), 𝒙 ∈ 𝔻𝑡𝑒𝑠𝑡去预测新
样本。神经网络属于机器学习的一个研究分支，它特指利用多个神经元去参数化映射函数
𝑓𝜃的模型。

感知机模型的结构如图 6.1 所示，它接受长度为𝑛的一维向量𝒙 = [𝑥1, 𝑥2, … , 𝑥𝑛]，每个  
输入节点通过权值为𝑤𝑖, 𝑖𝜖[1, 𝑛]的连接汇集为变量𝑧，即：  

𝑧 = 𝑤1 𝑥1 + 𝑤2 𝑥2 + ⋯ + 𝑤𝑛 𝑥𝑛 + b

# 6.2 全连接层

感知机模型的不可导特性严重约束了它的潜力，使得它只能解决极其简单的任务。实
际上，现代深度学习动辄数百万甚至上亿的参数规模，但它的核心结构与感知机并没有多
大差别。它在感知机的基础上，将不连续的阶跃激活函数换成了其它平滑连续可导的激活
函数，并通过堆叠多个网络层来增强网络的表达能力。
本节我们通过替换感知机的激活函数，同时并行堆叠多个神经元来实现多输入、多输
出的网络层结构。如图 6.4 所示，并行堆叠了 2 个神经元，即 2 个替换了激活函数的感知
机，构成 3 输入节点、2 个输出节点的网络层。

## 6.2.1 张量方式实现

在 TensorFlow 中，要实现全连接层，只需要定义好权值张量𝑾和偏置张量𝒃，并利用
TensorFlow 提供的批量矩阵相乘函数 tf.matmul()即可完成网络层的计算。  
例如，创建输入𝑿矩阵为𝑏 = 2个样本，每个样本的输入**特征长度**为𝑑in = 784，输出节点数为𝑑out = 256，故
定义权值矩阵𝑾的 shape 为[784,256]，并采用正态分布初始化𝑾；偏置向量𝒃的 shape 定义
为[256]，在计算完𝑿@𝑾后相加即可，最终全连接层的输出𝑶的 shape 为[2,256]，即 2 个样
本的特征，每个特征长度为 256，代码实现如下

In [114]:
import tensorflow as tf

# 创建 W,b 张量
x = tf.random.truncated_normal([2, 784])

w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros([256]))

o1 = tf.matmul(x, w1) + b1  # 线性变换
o1 = tf.nn.relu(o1)
o1.shape

TensorShape([2, 256])

## 6.2.2 层方式实现 layers.Dense(units, activation)  
全连接层本质上是矩阵的相乘和相加运算，实现并不复杂。但是作为最常用的网络层
之一，TensorFlow 中有更高层、使用更方便的层实现方式：layers.Dense(units, activation)。
通过 layer.Dense 类，只需要指定输出节点数 Units 和激活函数类型 activation 即可。需要注
意的是，输入节点数会根据第一次运算时的输入 shape 确定，同时根据输入、输出节点数
自动创建并初始化权值张量𝑾和偏置张量𝒃，因此在新建类 Dense 实例时，并不会立即创
建权值张量𝑾和偏置张量𝒃，而是需要调用 build 函数或者直接进行一次前向计算，才能完
成网络参数的创建。其中 activation 参数指定当前层的激活函数，可以为常见的激活函数或
自定义激活函数，也可以指定为 None，即无激活函数

In [115]:
x = tf.random.normal([4, 28 * 28])
layers = tf.keras.layers
# 创建全连接层，指定输出节点数和激活函数
fc = layers.Dense(512, activation=tf.nn.relu)
h1 = fc(x)  # 通过 fc 类实例完成一次全连接层的计算，返回输出张量
h1

<tf.Tensor: shape=(4, 512), dtype=float32, numpy=
array([[0.68652713, 0.31421173, 0.215366  , ..., 0.        , 0.17979091,
        0.        ],
       [0.36700043, 0.        , 0.11067569, ..., 0.        , 0.47782436,
        1.0862482 ],
       [0.        , 0.        , 0.16179556, ..., 1.5968728 , 1.6251692 ,
        0.20771837],
       [0.        , 0.09513193, 0.        , ..., 0.611598  , 0.        ,
        0.        ]], dtype=float32)>

上述通过一行代码即可以创建一层全连接层 fc，并指定输出节点数为 512，输入的节点数
在fc(x)计算时自动获取，并创建内部权值张量𝑾和偏置张量𝒃。我们可以通过类内部的成
员名 kernel 和 bias 来获取权值张量𝑾和偏置张量𝒃对象

In [116]:
fc.kernel  # 获取 Dense 类的权值矩阵

<tf.Variable 'dense_36/kernel:0' shape=(784, 512) dtype=float32, numpy=
array([[ 0.03127156,  0.0421644 ,  0.05240376, ..., -0.03683732,
        -0.03931131,  0.00689398],
       [ 0.00612184, -0.03003868, -0.04643788, ..., -0.02541891,
        -0.06551164, -0.03711088],
       [ 0.04866473,  0.02932405, -0.03862834, ...,  0.00370856,
        -0.00357565,  0.01770825],
       ...,
       [-0.02405337, -0.06503832, -0.05754158, ..., -0.01699909,
        -0.04237256,  0.01768109],
       [-0.02576108,  0.04305389,  0.01930559, ...,  0.05007228,
         0.0514864 ,  0.02008508],
       [-0.02080001,  0.04679633,  0.00527156, ..., -0.03486654,
        -0.0206156 , -0.00438739]], dtype=float32)>

In [117]:
fc.bias  # 获取Dense类的偏置向量

<tf.Variable 'dense_36/bias:0' shape=(512,) dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0.,

可以看到，权值张量𝑾和偏置张量𝒃的 shape 和内容均符合我们的理解。  
在优化参数时，需要获得网络的所有**待优化的张量**参数列表，可以通过类的trainable_variables 来返回待优化参数列表，代码如下

In [118]:
fc.trainable_variables
# 返回待优化参数列表

[<tf.Variable 'dense_36/kernel:0' shape=(784, 512) dtype=float32, numpy=
 array([[ 0.03127156,  0.0421644 ,  0.05240376, ..., -0.03683732,
         -0.03931131,  0.00689398],
        [ 0.00612184, -0.03003868, -0.04643788, ..., -0.02541891,
         -0.06551164, -0.03711088],
        [ 0.04866473,  0.02932405, -0.03862834, ...,  0.00370856,
         -0.00357565,  0.01770825],
        ...,
        [-0.02405337, -0.06503832, -0.05754158, ..., -0.01699909,
         -0.04237256,  0.01768109],
        [-0.02576108,  0.04305389,  0.01930559, ...,  0.05007228,
          0.0514864 ,  0.02008508],
        [-0.02080001,  0.04679633,  0.00527156, ..., -0.03486654,
         -0.0206156 , -0.00438739]], dtype=float32)>,
 <tf.Variable 'dense_36/bias:0' shape=(512,) dtype=float32, numpy=
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

实际上，网络层除了保存了待优化张量列表 trainable_variables，还有部分层包含了不
参与梯度优化的张量，如后续介绍的 Batch Normalization 层，可以通过
non_trainable_variables 成员返回所有不需要优化的参数列表。如果希望获得所有参数列
表，可以通过类的 variables 返回所有内部张量列表，

In [119]:
fc.variables

[<tf.Variable 'dense_36/kernel:0' shape=(784, 512) dtype=float32, numpy=
 array([[ 0.03127156,  0.0421644 ,  0.05240376, ..., -0.03683732,
         -0.03931131,  0.00689398],
        [ 0.00612184, -0.03003868, -0.04643788, ..., -0.02541891,
         -0.06551164, -0.03711088],
        [ 0.04866473,  0.02932405, -0.03862834, ...,  0.00370856,
         -0.00357565,  0.01770825],
        ...,
        [-0.02405337, -0.06503832, -0.05754158, ..., -0.01699909,
         -0.04237256,  0.01768109],
        [-0.02576108,  0.04305389,  0.01930559, ...,  0.05007228,
          0.0514864 ,  0.02008508],
        [-0.02080001,  0.04679633,  0.00527156, ..., -0.03486654,
         -0.0206156 , -0.00438739]], dtype=float32)>,
 <tf.Variable 'dense_36/bias:0' shape=(512,) dtype=float32, numpy=
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

对于全连接层，内部张量都参与梯度优化，故 variables 返回的列表与 trainable_variables 相
同。  
利用网络层类对象进行前向计算时，只需要调用类的__call__方法即可，即写成 fc(x)
方式便可，它会自动调用类的__call__方法，在__call__方法中会自动调用 call 方法，这一
设定由 TensorFlow 框架自动完成，因此用户只需要将网络层的前向计算逻辑实现在 call 方
法中即可。对于全连接层类，在 call 方法中实现𝜎(𝑿@𝑾 + 𝒃)的运算逻辑，非常简单，最
后返回全连接层的输出张量即可。

# 6.3 神经网络  
通过层层堆叠图 6.4 中的全连接层，保证前一层的输出节点数与当前层的输入节点数
匹配，，即可堆叠出任意层数的网络。我们把这种由神经元相互连接而成的网络叫做神经网
络。如图 6.5 所示，通过堆叠 4 个全连接层，可以获得层数为 4 的神经网络，由于每层均为全连接层，称为全连接网络。其中第 1~3 个全连接层在网络中间，称之为隐藏层 1、2、
3，最后一个全连接层的输出作为网络的输出，称为输出层。隐藏层 1、2、3 的输出节点数
分别为[256,128,64]，输出层的输出节点数为 10。
在设计全连接网络时，网络的结构配置等超参数可以按着经验法则自由设置，只需要
遵循少量的约束即可。例如，隐藏层 1 的输入节点数需和数据的实际特征长度匹配，每层
的输入层节点数与上一层输出节点数匹配，输出层的激活函数和节点数需要根据任务的具
体设定进行设计。总的来说，神经网络模型的结构设计自由度较大，如图 6.5 层中每层的
输出节点数不一定要
[512,64,32,10]等都是可行的。
设计为[256
至于与哪一组超参数是最优的，这需要很多的
,128,64,10]，可以自由搭配，如[256,256,64,10
领域经验
]或
知识
和大量的实验尝试，或者可以通过 AutoML 技术搜索出较优设定

## 6.3.1 张量方式实现

对于多层神经网络，以图 6.5 网络结构为例，需要分别定义各层的权值矩阵𝑾和偏置
向量𝒃。有多少个全连接层，则需要相应地定义数量相当的𝑾和𝒃，并且每层的参数只能用
于对应的层，不能混淆使用。图 6.5 的网络模型实现如下：

In [120]:
# 隐藏层1张量
w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros([256]))
# 隐藏层2张量
w2 = tf.Variable(tf.random.truncated_normal([256, 128], stddev=0.1))
b2 = tf.Variable(tf.zeros([128]))
# 隐藏层3张量
w3 = tf.Variable(tf.random.truncated_normal([128, 64], stddev=0.1))
b3 = tf.Variable(tf.zeros([64]))
# 输出层张量
w4 = tf.Variable(tf.random.truncated_normal([64, 10], stddev=0.1))
b4 = tf.Variable(tf.zeros([10]))

在计算时，只需要按照网络层的顺序，将上一层的输出作为当前层的输入即可，重复
直至最后一层，并将输出层的输出作为网络的输出，代码如下：

In [121]:
with tf.GradientTape() as tape:  # 梯度记录器
    # x: [b, 28*28]
    # 隐藏层 1 前向计算，[b, 28*28] => [b, 256]
    h1 = x @ w1 + tf.broadcast_to(b1, [x.shape[0], 256])
    h1 = tf.nn.relu(h1)
    # 隐藏层 2 前向计算，[b, 256] => [b, 128]
    h2 = h1 @ w2 + tf.broadcast_to(b2, [h1.shape[0], 128])
    h2 = tf.nn.relu(h2)
    # 隐藏层 3 前向计算，[b, 128] => [b, 64]
    h3 = h2 @ w3 + tf.broadcast_to(b3, [h2.shape[0], 64])
    h3 = tf.nn.relu(h3)
    # 输出层前向计算，[b, 64] => [b, 10]
    h4 = h3 @ w4 + b4

最后一层是否需要添加激活函数通常视具体的任务而定，这里加不加都可以。
在使用 TensorFlow 自动求导功能计算梯度时，需要将前向计算过程放置在
tf.GradientTape()环境中，从而利用 GradientTape 对象的 gradient()方法自动求解参数的梯
度，并利用 optimizers 对象更新参数

## 6.3.2 层方式实现  
对于常规的网络层，通过层方式实现起来更加简洁高效。首先新建各个网络层类，并指定各层的激活函数类型

In [122]:
Sequential = tf.keras.Sequential

fc1 = layers.Dense(256, activation=tf.nn.relu)  # 隐藏层1
fc2 = layers.Dense(128, activation=tf.nn.relu)  # 隐藏层 2
fc3 = layers.Dense(64, activation=tf.nn.relu)  # 隐藏层 3
fc4 = layers.Dense(10, activation=None)  # 输出层

在前向计算时，依序通过各个网络层即可，代码如下：

In [123]:
x = tf.random.normal([4, 28 * 28])
h1 = fc1(x)  # 通过隐藏层1得到输出
h2 = fc2(h1)  # 通过隐藏层2得到输出
h3 = fc3(h2)  # ...
h4 = fc4(h3)  # 通过输出层得到网络输出
h4.shape

TensorShape([4, 10])

对于这种数据依次向前传播的网络，也可以通过 *Sequential* 容器封装成一个网络大类对象，  
调用大类的前向计算函数一次即可完成所有层的前向计算，使用起来更加方便，实现如下

In [124]:
Sequential = tf.keras.Sequential

# 通过 Sequential 容器封装为一个网络类
model = Sequential(
    [
        layers.Dense(256, activation=tf.nn.relu),
        layers.Dense(128, activation=tf.nn.relu),
        layers.Dense(64, activation=tf.nn.relu),
        layers.Dense(10, activation=None)
    ]
)
# 前向计算时只需要调用一次网络大类对象，即可完成所有层的按序计算
out = model(x)
out.shape

TensorShape([4, 10])

## 6.3.3 优化目标  


# 6.4 激活函数

## 6.4.1 Sigmoid  
它的一个优良特性就是能够把𝑥 ∈ 𝑅的输入“压缩”到𝑥 ∈ (0,1)区间，这个区间的数值在机
器学习常用来表示以下意义  
在 TensorFlow 中，可以通过 tf.nn.sigmoid 实现 Sigmoid 函数

In [125]:
x = tf.linspace(-6., 6., 10)
x

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-6.       , -4.6666665, -3.3333333, -2.       , -0.6666665,
        0.666667 ,  2.       ,  3.333334 ,  4.666667 ,  6.       ],
      dtype=float32)>

In [126]:
tf.nn.sigmoid(x)  # 通过 Sigmoid 函数

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.00247262, 0.00931596, 0.0344452 , 0.11920292, 0.33924365,
       0.6607564 , 0.8807971 , 0.96555483, 0.99068403, 0.99752736],
      dtype=float32)>

## 6.4.2 ReLU  
ReLU 对小于 0 的值全部抑制为 0；对于正数则直接输
出，这种单边抑制特性来源于生物学.

在 TensorFlow 中，可以通过 tf.nn.relu 实现 ReLU 函数，代码如下

In [127]:
tf.nn.relu(x)  # 通过relu激活函数

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.      , 0.      , 0.      , 0.      , 0.      , 0.666667,
       2.      , 3.333334, 4.666667, 6.      ], dtype=float32)>

## 6.4.3 LeakyReLU  
ReLU 函数在𝑥 < 0时导数值恒为 0，也可能会造成梯度弥散现象，为了克服这个问
题，LeakyReLU 函数被提出

In [128]:
tf.nn.leaky_relu(x, alpha=0.1)

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-0.6       , -0.46666667, -0.33333334, -0.2       , -0.06666666,
        0.666667  ,  2.        ,  3.333334  ,  4.666667  ,  6.        ],
      dtype=float32)>

## 6.4.4 Tanh  
Tanh 函数能够将𝑥 ∈ 𝑅的输入“压缩”到(−1,1)区间，定义为：
$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} \\
= 2 \cdot \text{sigmoid}(2x) - 1
$$

In [129]:
tf.nn.tanh(x)

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-0.9999876 , -0.99982315, -0.99745804, -0.9640276 , -0.58278286,
        0.58278316,  0.9640276 ,  0.99745804,  0.99982315,  0.99998784],
      dtype=float32)>

# 6.5 输出层设计

我们来特别地讨论网络的最后一层的设计，它除了和所有的隐藏层一样，完成维度变
换、特征提取的功能，还作为输出层使用，需要根据具体的任务场景来决定是否使用激活
函数，以及使用什么类型的激活函数等。
我们将根据输出值的区间范围来分类讨论。常见的几种输出类型包括：
- ❑ 𝑜𝑖 ∈ 𝑅
𝑑 输出属于整个实数空间，或者某段普通的实数空间，比如函数值趋势的预
测，年龄的预测问题等。
- ❑ 𝑜𝑖 ∈ [0,1] 输出值特别地落在[0, 1]的区间，如图片生成，图片像素值一般用[0, 1]区间
的值表示；或者二分类问题的概率，如硬币正反面的概率预测问题。
- ❑ 𝑜𝑖 ∈ [0,  1], 𝑖 𝑜𝑖 = 1 输出值落在[0,1]的区间，并且所有输出值之和为 1，常见的如
多分类问题，如 MNIST 手写数字图片识别，图片属于 10 个类别的概率之和应为 1。
- ❑ 𝑜𝑖 ∈ [−1,  1] 输出值在[-1, 1]之间

## 6.5.1 普通实数空间

这一类问题比较普遍，像正弦函数曲线预测、年龄的预测、股票走势的预测等都属于
整个或者部分连续的实数空间，输出层可以不加激活函数。误差的计算直接基于最后一层
的输出𝒐和真实值𝒚进行计算，如采用均方差误差函数度量输出值𝒐与真实值𝒚之间的距离

## 6.5.2 [0, 1]区间  
输出值属于[0,1]区间也比较常见，比如图片的生成、二分类问题等。在机器学习中，
一般会将图片的像素值归一化到[0,1]区间，如果直接使用输出层的值，像素的值范围会分
布在整个实数空间。为了让像素的值范围映射到[0,1]的有效实数空间，需要在输出层后添
加某个合适的激活函数𝜎，其中 Sigmoid 函数刚好具有此功能。

## 6.5.3 [0,1]区间，和为 1  
输出值𝑜𝑖 ∈ [0,1]，且所有输出值之和为 1，这种设定以多分类问题最为常见。如图
6.15 所示，输出层的每个输出节点代表了一种类别，图中网络结构用于处理 3 分类任务，3
个节点的输出值分布代表了当前样本属于类别 A、类别 B 和类别 C 的概率𝑃(A|𝒙)、
, 𝑃(B|𝒙)、𝑃(C|𝒙)，考虑多分类问题中的样本只可能属于所有类别中的某一种，因此满足所
有类别概率之和为 1 的约束。

在 TensorFlow 中，可以通过 tf.nn.softmax 实现 Softmax 函数，代码如下:

In [130]:
z = tf.constant([2., 1., 0.1])
tf.nn.softmax(z)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([0.6590012 , 0.24243298, 0.09856589], dtype=float32)>

与 Dense 层类似，Softmax 函数也可以作为网络层类使用，通过类 layers.Softmax(axis=-1)
可以方便添加 Softmax 层，其中 axis 参数指定需要进行计算的维度。
在 Softmax 函数的数值计算过程中，容易因输入值偏大发生数值溢出现象；在计算交
叉熵时，也会出现数值溢出的问题。为了数值计算的稳定性，TensorFlow 中提供了一个统
一的接口，将 Softmax 与交叉熵损失函数同时实现，同时也处理了数值不稳定的异常，一
般推荐使用这些接口函数，避免分开使用 Softmax 函数与交叉熵损失函数。函数式接口为
tf.keras.losses.categorical_crossentropy(y_true, y_pred, from_logits=False)，其中 y_true 代表了
One-hot 编码后的真实标签，y_pred 表示网络的预测值，当 from_logits 设置为 True 时，
y_pred 表示须为未经过 Softmax 函数的变量 z；当 from_logits 设置为 False 时，y_pred 表示
为经过 Softmax 函数的输出。为了数值计算稳定性，一般设置 from_logits 为 True，此时
tf.keras.losses.categorical_crossentropy 将在内部进行 Softmax 函数计算，所以不需要在模型
中显式调用 Softmax 函数，例如

In [131]:
z = tf.random.normal([2, 10])  # 构造输出层的输出
y_onehot = tf.constant([1, 3])  # 构造真实值
y_onehot = tf.one_hot(y_onehot, depth=10)  # one-hot 编码
# 输出层未使用 Softmax 函数，故 from_logits 设置为 True
# 这样 categorical_crossentropy 函数在计算损失函数前，会先内部调用 Softmax 函数
loss = tf.keras.losses.categorical_crossentropy(y_onehot, z, from_logits=True)
loss = tf.reduce_mean(loss)
loss

<tf.Tensor: shape=(), dtype=float32, numpy=2.6127086>

除了函数式接口，也可以利用 losses.CategoricalCrossentropy(from_logits)类方式同时实
现 Softmax 与交叉熵损失函数的计算，from_logits 参数的设置方式相同。例如：

In [132]:
# 创建 Softmax 与交叉熵计算类，输出层的输出 z 未使用 softmax
from tensorflow import keras
criteon = keras.losses.CategoricalCrossentropy(from_logits=True)
loss = criteon(y_onehot, z)  # 计算损失
loss

<tf.Tensor: shape=(), dtype=float32, numpy=2.6127086>

## 6.5.4 [-1, 1]  
如果希望输出值的范围分布在(−1,1)区间，可以简单地使用 tanh 激活函数，实现如
下

In [133]:
x = tf.linspace(-6., 6., 10)
tf.tanh(x)

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-0.9999876 , -0.99982315, -0.99745804, -0.9640276 , -0.58278286,
        0.58278316,  0.9640276 ,  0.99745804,  0.99982315,  0.99998784],
      dtype=float32)>

输出层的设计具有一定的灵活性，可以根据实际的应用场景自行设计，充分利用现有激活
函数的特性

# 6.6 误差计算  
在搭建完模型结构后，下一步就是选择合适的误差函数来计算误差。常见的误差函数
有均方差、交叉熵、KL 散度、Hinge Loss 函数等，其中均方差函数和交叉熵函数在深度学
习中比较常见，均方差函数主要用于回归问题，交叉熵函数主要用于分类问题。

## 6.6.1 均方差误差函数  
均方差(Mean Squared Error，简称 MSE)误差函数把输出向量和真实向量映射到笛卡尔
坐标系的两个点上，通过计算这两个点之间的欧式距离(准确地说是欧式距离的平方)来衡
量两个向量之间的差距：
$$
\text{MSE}(\mathbf{y}, \mathbf{o}) \triangleq \frac{1}{d_{\text{out}}} \sum_{i=1}^{d_{\text{out}}} (y_i - o_i)^2
$$

MSE 误差函数的值总是大于等于 0，当 MSE 函数达到最小值 0 时，输出等于真实标签，
此时神经网络的参数达到最优状态。
均方差误差函数广泛应用在回归问题中，实际上，分类问题中也可以应用均方差误差
函数。在 TensorFlow 中，可以通过函数方式或层方式实现 MSE 误差计算。例如，使用函
数方式实现 MSE 计算，代码如下

In [134]:
o = tf.random.normal([2, 10])  # 构造网络输出
y_onehot = tf.constant([1, 3])  # 构造真实值
y_onehot = tf.one_hot(y_onehot, depth=10)
loss = keras.losses.MSE(y_onehot, o)  # 计算均方差
loss

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([0.97682846, 0.65791214], dtype=float32)>

特别要注意的是，MSE 函数返回的是每个样本的均方差，需要在样本维度上再次平均来获
得平均样本的均方差，实现如下：

In [135]:
loss = tf.reduce_mean(loss)
loss

<tf.Tensor: shape=(), dtype=float32, numpy=0.8173703>

也可以通过层方式实现，对应的类为 keras.losses.MeanSquaredError()，和其他层的类一
样，调用__call__函数即可完成前向计算，代码如下：

In [136]:
criteon = keras.losses.MeanSquaredError()
loss = criteon(y_onehot, o)
loss

<tf.Tensor: shape=(), dtype=float32, numpy=0.8173703>

## 6.6.2 交叉熵误差函数  
在介绍交叉熵损失函数之前，我们首先来介绍信息学中熵(Entropy)的概念。1948 年，
Claude Shannon 将热力学中熵的概念引入到信息论中，用来衡量信息的不确定度。熵在信
息学科中也叫信息熵，或者香农熵。熵越大，代表不确定性越大，信息量也就越大。某个
分布𝑃(𝑖)的熵定义为:
$$
H(P) \triangleq -\sum_{i} P(i) \log_2 P(i)
$$

在介绍完熵的概念后，我们基于熵引出交叉熵(Cross Entropy)的定义：
$$ H(p||q) \triangleq -\sum_i p(i) \log_2 q(i) $$

# 6.7 神经网络类型  
全连接层是神经网络最基本的网络类型，对后续神经网络类型的研究有巨大的贡献，
全连接层前向计算流程相对简单，梯度求导也较简单，但是它有一个最大的缺陷，在处理
较大特征长度的数据时，全连接层的参数量往往较大，使得深层数的全连接网络参数量巨
大，训练起来比较困难。近年来，社交媒体的发达产生了海量的图片、视频、文本等数字
资源，极大地促进了神经网络在计算机视觉、自然语言处理等领域中的研究，相继提出了
一系列的神经网络变种类型


## 6.7.1 卷积神经网络  
如何识别、分析并理解图片、视频等数据是计算机视觉的一个核心问题，全连接层在
处理高维度的图片、视频数据时往往出现网络参数量巨大，训练非常困难的问题。通过利
用局部相关性和权值共享的思想，Yann Lecun 在 1986 年提出了卷积神经网络
(Convolutional Neural Network，简称 CNN)。随着深度学习的兴盛，卷积神经网络在计算机
视觉中的表现大大地超越了其它算法模型，呈现统治计算机视觉领域之势。这其中比较流
行的模型有用于图片分类的 AlexNet、VGG、GoogLeNet、ResNet、DenseNet 等，用于目
标识别的 RCNN、Fast RCNN、Faster RCNN、Mask RCNN、YOLO、SSD 等。我们将在第
10 章详细介绍卷积神经网络原理。

## 6.7.2 循环神经网络  
除了具有空间结构的图片、视频等数据外，序列信号也是非常常见的一种数据类型，
其中一个最具代表性的序列信号就是文本数据。如何处理并理解文本数据是自然语言处理
的一个核心问题。卷积神经网络由于缺乏 Memory 机制和处理不定长序列信号的能力，并
不擅长序列信号的任务。循环神经网络(Recurrent Neural Network，简称 RNN)在 Yoshua 
Bengio、Jürgen Schmidhuber 等人的持续研究下，被证明非常擅长处理序列信号。1997
年，Jürgen Schmidhuber 提出了 LSTM 网络，作为 RNN 的变种，它较好地克服了 RNN 缺
乏长期记忆、不擅长处理长序列的问题，在自然语言处理中得到了广泛的应用。基于
LSTM 模型，Google 提出了用于机器翻译的 Seq2Seq 模型，并成功商用于谷歌神经机器翻
译系统(GNMT)。其他的 RNN 变种还有 GRU、双向 RNN 等。我们将在第 11 章详细介绍
循环神经网络原理。

## 6.7.3 注意力(机制)网络  
RNN 并不是自然语言处理的最终解决方案，近年来随着注意力机制(Attention
Mechanism)的提出，克服了 RNN 训练不稳定、难以并行化等缺陷，在自然语言处理和图
片生成等领域中逐渐崭露头角。注意力机制最初在图片分类任务上提出，但逐渐开始侵蚀
NLP 各大任务。2017 年，Google 提出了第一个利用纯注意力机制实现的网络模型
Transformer，随后基于 Transformer 模型相继提出了一系列的用于机器翻译的注意力网络模
型，如 GPT、BERT、GPT-2 等。在其它领域，基于注意力机制，尤其是自注意力(SelfAttention)机制构建的网络也取得了不错的效果，比如基于自注意力机制的 BigGAN 模型
等。

## 6.7.4 图卷积神经网络  
图片、文本等数据具有规则的空间、时间结构，称为 Euclidean Data(欧几里德数据)。
卷积神经网络和循环神经网络被证明非常擅长处理这种类型的数据。而像类似于社交网
络、通信网络、蛋白质分子结构等一系列的不规则空间拓扑结构的数据，它们显得力不从
心。2016 年，Thomas Kipf 等人基于前人在一阶近似的谱卷积算法上提出了图卷积网络
(Graph Convolution Network，GCN)模型。GCN 算法实现简单，从空间一阶邻居信息聚合的角度也能直观地理解，在半监督任务上取得了不错效果。随后，一系列的网络模型相继
被提出，如 GAT，EdgeConv，DeepGCN 等。

# 6.8 汽车油耗预测实战 
本节我们将利用全连接网络模型来完成汽车的效能指标 MPG(Mile Per Gallon，每加仑
燃油英里数)的预测问题实战。

## 6.8.1 数据集  
我们采用 Auto MPG 数据集，它记录了各种汽车效能指标与气缸数、重量、马力等其
它因子的真实数据，查看数据集的前 5 项，如表 6.1 所示，其中每个字段的含义列在表
6.2 中。除了产地的数字字段表示类别外，其他字段都是数值类型。对于产地地段，1 表示
美国，2 表示欧洲，3 表示日本。

表 6.1 Auto MPG 数据集前 5 项

| MPG  | Cylinders | Displacement | Horsepower | Weight  | Acceleration | Model Year | Origin |
|------|-----------|--------------|------------|---------|--------------|------------|--------|
| 18.0 | 8         | 307.0        | 130.0      | 3504.0  | 12.0         | 70         | 1      |
| 15.0 | 8         | 350.0        | 165.0      | 3693.0  | 11.5         | 70         | 1      |
| 18.0 | 8         | 318.0        | 150.0      | 3436.0  | 11.0         | 70         | 1      |
| 16.0 | 8         | 304.0        | 150.0      | 3433.0  | 12.0         | 70         | 1      |
| 17.0 | 8         | 302.0        | 140.0      | 3449.0  | 10.5         | 70         | 1      |

表 6.2 数据集字段含义

| MPG       | Cylinders | Displacement | Horsepower | Weight | Acceleration | Model Year | Origin |
|-----------|-----------|--------------|------------|--------|--------------|------------|--------|
| 每加仑燃油英里 | 气缸数     | 排量          | 马力        | 重量    | 加速度        | 型号年份     | 产地    |

Auto MPG 数据集一共记录了 398 项数据，我们从 UCI 服务器下载并读取数据集到
DataFrame 对象中，代码如下：

In [137]:
# 在线下载汽车效能数据集
import pandas as pd
dataset_path = keras.utils.get_file("auto-mpg.data",
                                    "http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data")
# 利用 pandas 读取数据集，字段有效能（公里数每加仑），气缸数，排量，马力，重量
# 加速度，型号年份，产地
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']
raw_dataset = pd.read_csv(dataset_path, names=column_names,
                          na_values="?", comment='\t',
                          sep=" ", skipinitialspace=True)
dataset = raw_dataset.copy()

#  查看部分数据
dataset.head()
# 原始表格中的数据可能含有空字段(缺失值)的数据项，需要清除这些记录项：
print('含有空字段(缺失值)的数据项\n', dataset.isna().sum())
dataset = dataset.dropna()  # 删除空白数据项
dataset.isna().sum()

含有空字段(缺失值)的数据项
 MPG             0
Cylinders       0
Displacement    0
Horsepower      6
Weight          0
Acceleration    0
Model Year      0
Origin          0
dtype: int64


MPG             0
Cylinders       0
Displacement    0
Horsepower      0
Weight          0
Acceleration    0
Model Year      0
Origin          0
dtype: int64

In [138]:
origin = dataset.pop("Origin")

In [139]:
# 根据 origin 列来写入新的 3 个列
dataset["USA"] = (origin == 1) * 1.0
dataset["Europe"] = (origin == 2) * 1.0
dataset["Japan"] = (origin == 3) * 1.0

dataset.tail()  # 查看新表格的后几项

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,USA,Europe,Japan
393,27.0,4,140.0,86.0,2790.0,15.6,82,1.0,0.0,0.0
394,44.0,4,97.0,52.0,2130.0,24.6,82,0.0,1.0,0.0
395,32.0,4,135.0,84.0,2295.0,11.6,82,1.0,0.0,0.0
396,28.0,4,120.0,79.0,2625.0,18.6,82,1.0,0.0,0.0
397,31.0,4,119.0,82.0,2720.0,19.4,82,1.0,0.0,0.0


按着 8:2 的比例切分数据集为训练集和测试集

In [140]:
# 切分为训练集和测试集
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

将 MPG 字段移出为标签数据：

In [141]:
# 移动 MPG 油耗效能这一列为真实标签 Y
train_labels = train_dataset.pop("MPG")
test_labels = test_dataset.pop("MPG")

统计训练集的各个字段数值的均值和标准差，并完成数据的标准化，通过 norm()函数
实现，代码如下

In [149]:
# 查看训练集的输入 X 的统计数据
train_stats = train_dataset.describe()
train_stats = train_stats.transpose()
train_stats

,count,mean,std,min,25%,50%,75%,max
Cylinders,314.0,5.477707,1.699788,3.0,4.00,4.0,8.00,8.0
Displacement,314.0,195.318471,104.331589,68.0,105.50,151.0,265.75,455.0
Horsepower,314.0,104.869427,38.096214,46.0,76.25,94.5,128.00,225.0
Weight,314.0,2990.251592,843.898596,1649.0,2256.50,2822.5,3608.00,5140.0
Acceleration,314.0,15.559236,2.789230,8.0,13.80,15.5,17.20,24.8
Model Year,314.0,75.898089,3.675642,70.0,73.00,76.0,79.00,82.0
USA,314.0,0.624204,0.485101,0.0,0.00,1.0,1.00,1.0
Europe,314.0,0.178344,0.383413,0.0,0.00,0.0,0.00,1.0
Japan,314.0,0.197452,0.398712,0.0,0.00,0.0,0.00,1.0
